# Transformer Block From Scratch

This notebook expands the raw-attention view into the full decoder-only transformer block. The goal is to keep the tensor mechanics visible long enough that the package abstraction in `TinyGPT` feels earned rather than magical.


## Problem: What Must This Component Compute?

At this point in the model, the input has shape `(B, T, C)`: batch size, sequence length, and channel dimension. The component must transform that tensor without leaking information from future token positions, while preserving a residual pathway that makes deep optimization practical.

We will first write the operation directly with tensors so every dimension is visible. Only after the numerical checks pass will we wrap the operation in a reusable module.

Why this matters later: when an architecture paper claims to improve attention, memory, quantization, or world modeling, it is usually changing one of these constraints: what information can move across positions, how much memory is required, what objective is optimized, or which representations are preserved.


## 1. Residual stream.

The residual stream is the running state of the model. Each block reads `x`, computes an update `Δx`, and writes back `x + Δx`. This preserves a direct path for gradients and lets later layers refine rather than replace earlier features.


## 2. Layer normalization.

Layer normalization standardizes each token representation independently across channels:

\[
\text{LN}(x_{t}) = \frac{x_t - \mu_t}{\sqrt{\sigma_t^2 + \epsilon}}.
\]

Here `\mu_t` and `\sigma_t^2` are the mean and variance across the channel dimension for token position `t`. In GPT-style pre-norm blocks, normalization happens before each learned sublayer so the residual path stays clean.


In [ ]:
import torch
import torch.nn.functional as F

from llm_from_scratch.configs import ModelConfig, profile_config, set_seed
from llm_from_scratch.model import TinyGPT, count_parameters

set_seed(123)
B, T, C = 2, 4, 8
residual = torch.randn(B, T, C)
update = 0.1 * torch.randn(B, T, C)
residual_stream = residual + update

mean = residual_stream.mean(dim=-1, keepdim=True)
var = residual_stream.var(dim=-1, keepdim=True, unbiased=False)
manual_ln = (residual_stream - mean) / torch.sqrt(var + 1e-5)
torch_ln = F.layer_norm(residual_stream, (C,))

assert manual_ln.shape == residual_stream.shape
assert torch.allclose(manual_ln, torch_ln, atol=1e-5)

{
    "residual_shape": tuple(residual_stream.shape),
    "per_token_mean": manual_ln.mean(dim=-1),
    "per_token_var": manual_ln.var(dim=-1, unbiased=False),
}


## 3. Multi-head causal self-attention module.

The attention sublayer forms queries, keys, and values, splits channels into heads, masks future positions, and mixes earlier token information:

\[
Q = XW_Q, \quad K = XW_K, \quad V = XW_V,
\]
\[
\text{Attn}(X) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_h}} + M\right)V.
\]

Multi-head structure lets different subspaces attend to different token relationships while preserving the same residual width `C` after concatenation and projection.


## 4. Feed-forward network.

The feed-forward network then applies the same channel-mixing MLP to every token independently. In GPT-style blocks the hidden size is usually `4C`, giving the MLP more capacity than the residual width alone. Attention moves information across positions; the feed-forward network remaps features within each position.


In [ ]:
from llm_from_scratch.functional import causal_mask, stable_softmax

set_seed(123)
B, T, C, H = 2, 4, 8, 2
head_dim = C // H
x = torch.randn(B, T, C)

W_q = torch.randn(C, C)
W_k = torch.randn(C, C)
W_v = torch.randn(C, C)
W_1 = torch.randn(C, 4 * C)
b_1 = torch.randn(4 * C)
W_2 = torch.randn(4 * C, C)
b_2 = torch.randn(C)

q = (x @ W_q).view(B, T, H, head_dim).transpose(1, 2)
k = (x @ W_k).view(B, T, H, head_dim).transpose(1, 2)
v = (x @ W_v).view(B, T, H, head_dim).transpose(1, 2)

scores = q @ k.transpose(-2, -1) / torch.sqrt(torch.tensor(float(head_dim)))
mask = causal_mask(T).view(1, 1, T, T)
masked_scores = scores.masked_fill(~mask, float('-inf'))
weights = stable_softmax(masked_scores, dim=-1)
attn_out = (weights @ v).transpose(1, 2).contiguous().view(B, T, C)

ff_hidden = F.gelu(x @ W_1 + b_1)
ff_out = ff_hidden @ W_2 + b_2

future_weights = torch.triu(weights[0, 0], diagonal=1)
assert attn_out.shape == x.shape
assert ff_out.shape == x.shape
assert torch.allclose(future_weights, torch.zeros_like(future_weights), atol=1e-6)
assert torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)), atol=1e-6)

{
    "attention_shape": tuple(attn_out.shape),
    "ff_shape": tuple(ff_out.shape),
    "head0_future_weights": future_weights,
}


## 5. Transformer block.

A pre-norm GPT block composes the two sublayers with residual additions:

\[
x' = x + \text{Attn}(\text{LN}_1(x)), \qquad
y = x' + \text{FFN}(\text{LN}_2(x')).
\]

This is the smallest reusable architectural unit in the decoder stack.


## 6. Full decoder-only GPT.

Stacking `n_layer` transformer blocks after token and position embeddings gives the full decoder-only model. The final layer normalization and tied output head turn hidden states back into logits over the vocabulary.


In [ ]:
import torch
from llm_from_scratch.configs import ModelConfig
from llm_from_scratch.model import TinyGPT, count_parameters

cfg = ModelConfig(vocab_size=64, block_size=16, n_embd=32, n_head=4, n_layer=2)
model = TinyGPT(cfg)
idx = torch.randint(0, cfg.vocab_size, (2, cfg.block_size))
logits, loss = model(idx, idx)
logits.shape, loss.item(), count_parameters(model)


## 7. Parameter count and scaling intuition.

Parameter count grows roughly linearly with layer count and roughly quadratically with embedding width because projection matrices are `C x C` or `C x 4C`. That is why widening a model becomes expensive quickly even before longer contexts increase activation and attention costs.


In [ ]:
quick_cfg, _ = profile_config('quick')
study_cfg, _ = profile_config('study')

quick_model = TinyGPT(quick_cfg)
study_model = TinyGPT(study_cfg)

quick_params = count_parameters(quick_model)
study_params = count_parameters(study_model)

assert study_params > quick_params
assert logits.shape == (2, cfg.block_size, cfg.vocab_size)

{
    "quick_params": quick_params,
    "study_params": study_params,
    "scaling_ratio": round(study_params / quick_params, 2),
}


## Abstraction Bridge

The raw tensor code above matches the structure of the package modules. `CausalSelfAttention` handles QKV projection, masking, and projection back to the residual width; `GPTBlock` adds the pre-norm residual pattern; `TinyGPT` adds embeddings, a stack of blocks, final normalization, and the tied output head. When these abstractions are trustworthy, architecture changes become controlled swaps of known pieces instead of opaque rewrites.


## Exercise Checkpoint 1

1. If `x` has shape `(B, T, C)` and `n_head = H`, what shape should `q.view(B, T, H, C // H).transpose(1, 2)` have?
2. Why does the residual stream make optimization easier than repeatedly overwriting the entire hidden state?


## Exercise Checkpoint 2

1. If you double `n_layer` while keeping `n_embd` fixed, which parameter groups scale most directly with depth?
2. Why does widening `n_embd` usually increase parameter count faster than increasing `n_layer` by the same multiplicative factor?
